# Data Description
- Author: 詹明翰、謝子詮、曾翊涵（共同作者不代表個別成員已完成工作）
- Date first created: 20261126
- Background: Unit2.2正式全部45,130候選事故之正式空間配對側表，91欄而非複製原191欄；包含候選JSON、幾何及保守互斥狀態、S0–S3／eligibility、區名／地址、距離與跨地理變化。未合格事故仍保留，僅規格ID為NA。 技術交付Unit2.3–2.7，不是正式Dataset A／B1／B2／B3／C或Finding。
- Unit of observation: 一個Unit2.1第一當事人操作型候選事故，不是官方唯一事故ID。
- Sample period: 2024–2025事故研究；主地理固定456里，歷史113／114年版僅單時點快照，不認證全年每日法定界。S0（主要）：固定界，within與covered_by各唯一且同里，來源區名可確認並一致，地址沒有明確空間衝突。S1：固定界幾何唯一，保留區名／地址衝突。S2：對應年度歷史界＋同S0保守條件。S3：對應年度歷史界＋同S1幾何條件。S0⊆S1、S2⊆S3；同家族共同有效ID必相同，敏感規格不自動升主分析。
- Known issues: 事故原datum未認證、4326僅工作假設；固定原幾何保留157對正面積交集、335個全市union孔洞（個別面內環總數1），沒有snap／buffer修補／填洞／去重。全45,130事故含三筆缺座標未補值；原區名／地址不更正，無最近里／地理編碼。糖廍里同規則納入，原糖?／糖蔀名及provisional_match保留。觀測配對0不等於現實無事故，未制定低樣本排除政策。人工DA／PM驗收仍待完成。
- Definition for each variable: 下方完整91欄字典：dtype、單位、來源、NA及品質；保存於schema，正式欄序與實際reader一致。候選碼陣列使用排序JSON字串，[]表示查無命中，NA表示未查；within／covered_by候選完整保存，不取多重第一筆。
- Data path: `data/processed/taipei_crash_spatial_assignment_2024_2025.csv`；45,130列×91欄、40,948,727 bytes。
- Source data: schema.source_inputs精確列出19個既有必要檔及本次helper之固定SHA；事故觀測只來自Unit2.1正式CSV，來源文件：[Unit2.1事故說明](taipei_crashes_2024_2025_standardized_documentation.ipynb)、[既有crosswalk說明](village_crosswalk_documentation.ipynb)、[歷史里界原始說明](../raw/unit17_historical_boundaries_raw_documentation.ipynb)。本文件獨立讀取保存CSV／schema，不使用處理Notebook記憶體，不重建raw事故版本或重跑Chapter1。
- Processing code: [prepare_taipei_village_base_and_crash_spatial_assignment.ipynb](../code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb)，SHA `ef8a3d21f38ecb902f9495720b0094363ebd57cf444b837176fe2b815992abc4`；[必要共用helper](../code/unit22_spatial_support.py)，SHA `06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc`。
- Role: spatial_assignment。
- Verification status: 以本文件乾淨kernel實際讀回、完整assertions及execution metadata為準；本輪Codex協助，成員驗收尚待完成。
- SHA-256: `2abcb5a69c8998c2f8b40c975404b93d629fd725899f74afc991f349d9201ba5`。
- Downstream analysis input: `data/processed/taipei_crash_spatial_assignment_2024_2025.csv`，與Data path相同。
- Schema / dictionary: [taipei_crash_spatial_assignment_2024_2025_schema.json](taipei_crash_spatial_assignment_2024_2025_schema.json)，SHA `d43e0e3bcc1af2b7683d9466ccb4c441c39a6d9ce56f957b0ed5a89c1b487ad3`。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。Actual Contribution：Codex本地協助；未推定成員修改、DA驗收或PM審查已完成。


In [1]:
from pathlib import Path
import sys,json,hashlib,platform
import pandas as pd
import numpy as np
from IPython.display import display,Markdown
for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():
        PROJECT_ROOT=p;break
else:raise FileNotFoundError('請從研究工作包內執行')
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert sha(PROJECT_ROOT/'data/code/unit22_spatial_support.py')=='06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit22_spatial_support import Context,show,load_csv,validate_saved_pair,STATES,GEOMETRY_STATES
pd.set_option('display.max_columns',None);pd.set_option('display.max_colwidth',110)
print('Python',platform.python_version(),'existing .venv / DSSI Urban Crash; helper SHA256','06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc')


Python 3.12.2 existing .venv / DSSI Urban Crash; helper SHA256 06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc


## Change for Your Own Data

按固定CSV、schema、producer及helper hash讀取，任何不同即停止。UTF-8-SIG、LF、__MISSING__；原空字串保留，不用預設NA／dtype推論。附件**MISSING**文字不覆蓋實際schema契約。兩資料獨立讀回並按鍵驗證；不執行Context處理流程。

In [2]:
EXPECTED_OUTPUT_HASHES={'data/processed/taipei_village_base.csv': 'db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209', 'data/processed/taipei_village_base_schema.json': 'a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070', 'data/processed/taipei_crash_spatial_assignment_2024_2025.csv': '2abcb5a69c8998c2f8b40c975404b93d629fd725899f74afc991f349d9201ba5', 'data/processed/taipei_crash_spatial_assignment_2024_2025_schema.json': 'd43e0e3bcc1af2b7683d9466ccb4c441c39a6d9ce56f957b0ed5a89c1b487ad3'}
EXPECTED_PRODUCER_SHA='ef8a3d21f38ecb902f9495720b0094363ebd57cf444b837176fe2b815992abc4'
assert sha(PROJECT_ROOT/'data/code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb')==EXPECTED_PRODUCER_SHA
for relative,expected in EXPECTED_OUTPUT_HASHES.items():assert sha(PROJECT_ROOT/relative)==expected,relative
base_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_village_base_schema.json').read_text(encoding='utf-8'))
assignment_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_crash_spatial_assignment_2024_2025_schema.json').read_text(encoding='utf-8'))
for relative,expected in base_contract['source_inputs'].items():assert sha(PROJECT_ROOT/relative)==expected,relative
assert base_contract['source_inputs']==assignment_contract['source_inputs']
base=load_csv(PROJECT_ROOT/base_contract['dataset_path'],base_contract)
assignment=load_csv(PROJECT_ROOT/assignment_contract['dataset_path'],assignment_contract)
upstream_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_crashes_2024_2025_standardized_schema.json').read_text(encoding='utf-8'))
upstream=load_csv(PROJECT_ROOT/'data/processed/taipei_crashes_2024_2025_standardized.csv',upstream_contract)
contract=assignment_contract
df=assignment


## Summary

In [3]:
print({'data_path':contract['dataset_path'],'rows':len(df),'columns':len(df.columns),'bytes':(PROJECT_ROOT/contract['dataset_path']).stat().st_size,'sha256':sha(PROJECT_ROOT/contract['dataset_path']),'role':contract['role']})
show('來源／程式固定版本',pd.DataFrame(list(contract['source_inputs'].items()),columns=['path','sha256']))
show('CSV與CRS契約',pd.DataFrame([{'csv':str(contract['csv']),'crs':str(contract['crs']),'id_namespaces':str(contract['id_namespaces'])}]).T)

{'data_path': 'data/processed/taipei_crash_spatial_assignment_2024_2025.csv', 'rows': 45130, 'columns': 91, 'bytes': 40948727, 'sha256': '2abcb5a69c8998c2f8b40c975404b93d629fd725899f74afc991f349d9201ba5', 'role': 'spatial_assignment'}


**來源／程式固定版本**

,path,sha256
0,data/processed/taipei_crashes_2024_2025_standardized.csv,b2ac0704068319b74f9aef2a28ad810d0cef473894a2a5cd23976f3fa7d2c252
1,data/processed/taipei_crashes_2024_2025_standardized_schema.json,c3f064b3572d940489bdc86bb5c49dfaea8a228a3e7addbc8f9544bad324b650
2,data/processed/taipei_crashes_2024_2025_standardized_documentation.ipynb,7d5dd73cc2f33308c7b4cdc63ff47839feb34c35baef830b70114ae5e4cc6e0b
3,data/code/prepare_taipei_crashes_2024_2025.ipynb,c14dbd34bd3d776339e292b820a049b792a30ec49f2a8fc0173440ed6e146a9e
4,data/raw/G97_A_CAVLGE_P.shp,f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc
5,data/raw/G97_A_CAVLGE_P.shx,ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e
6,data/raw/G97_A_CAVLGE_P.dbf,2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577
7,data/raw/G97_A_CAVLGE_P.prj,8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5
8,data/raw/G97_A_CAVLGE_P.cpg,3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824
9,data/raw/113村里界.zip,32d7f1d611f91b1c63bea7290dd4db93b919bf380cce92b634fca992c974dd94


,path,sha256
18,data/code/unit17_validation_support.py,1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080
19,data/code/unit22_spatial_support.py,06af690caf93af87b0f203d354fc894310aee87eca1d9f6f33df6930b13863fc


**CSV與CRS契約**

,0
csv,"{'encoding': 'utf-8-sig', 'missing_sentinel': '__MISSING__', 'line_terminator': 'LF', 'keep_default_na': F..."
crs,"{'fixed_source': 'EPSG:3826', 'history_source': 'EPSG:3824', 'accident_source': 'EPSG:4326 working assumpt..."
id_namespaces,"{'village_id': 'fixed CPTVID 10 digits, string', 'population_code': 'existing crosswalk 11 digits, string'..."


### Sample

保存正式資料的實際前十列，沒有以處理kernel物件代替磁碟輸入。

In [4]:
display(df.head(10))

,record_id,source_year,source_row_number,coordinate_x_raw,coordinate_y_raw,longitude_candidate,latitude_candidate,coordinate_parse_valid,coordinate_missing_flag,original_district_label_raw,original_district_name,original_district_readable,accident_address_raw,address_district_name,address_district_readable,raw_address_district_conflict,point_valid,point_easting_3826_m,point_northing_3826_m,fixed_geometry_status,fixed_geometry_unique,fixed_geometric_village_id,fixed_spatial_district,fixed_unique_original_boundary_code,fixed_within_candidate_count,fixed_within_candidate_ids,fixed_within_original_codes,fixed_covered_by_candidate_count,fixed_covered_by_candidate_ids,fixed_covered_by_original_codes,fixed_district_comparable,fixed_district_match,fixed_district_conflict,fixed_address_comparable,fixed_address_match,fixed_address_conflict,fixed_address_status,fixed_conservative_eligible,fixed_assignment_status,fixed_distance_to_original_district_m,fixed_distance_valid,historical_geometry_status,historical_geometry_unique,historical_geometric_village_id,historical_spatial_district,historical_unique_original_boundary_code,historical_within_candidate_count,historical_within_candidate_ids,historical_within_original_codes,historical_covered_by_candidate_count,historical_covered_by_candidate_ids,historical_covered_by_original_codes,historical_district_comparable,historical_district_match,historical_district_conflict,historical_address_comparable,historical_address_match,historical_address_conflict,historical_address_status,historical_conservative_eligible,historical_assignment_status,historical_distance_to_original_district_m,historical_distance_valid,eligible_s0,village_id_s0,eligible_s1,village_id_s1,eligible_s2,village_id_s2,eligible_s3,village_id_s3,historical_boundary_year,historical_boundary_code,fixed_historical_geometry_assignment_same,geometry_eligibility_changed,geometry_assigned_to_unassigned,geometry_unassigned_to_assigned,fixed_historical_conservative_assignment_same,conservative_eligibility_changed,conservative_assigned_to_unassigned,conservative_unassigned_to_assigned,geometric_village_changed,geometric_district_changed,geometry_status_changed,geometry_village_or_status_changed,district_or_assignment_availability_changed,district_conflict_to_consistent,district_consistent_to_conflict,new_district_conflict_including_unassigned,geometry_change_reason,conservative_change_reason
0,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000001,2024,1,121.5664082,25.0430901,121.566408,25.04309,True,False,07信義區,信義區,True,信義區基隆路1段147巷5弄11號,信義區,True,False,True,307153.886538,2770669.900836,unique_interior,True,6300200006,信義區,6300200006,1,"[""6300200006""]","[""6300200006""]",1,"[""6300200006""]","[""6300200006""]",True,True,False,True,True,False,consistent,True,conservative_success,0.0,True,unique_interior,True,6300200006,信義區,63000020006,1,"[""6300200006""]","[""63000020006""]",1,"[""6300200006""]","[""63000020006""]",True,True,False,True,True,False,consistent,True,conservative_success,0.0,True,True,6300200006,True,6300200006,True,6300200006,True,6300200006,2024,63000020006,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,unchanged,unchanged
1,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000003,2024,3,121.529308,25.0339234,121.529308,25.033923,True,False,05中正區,中正區,True,中正區信義路2段與信義路2段181巷口,中正區,True,False,True,303414.159021,2769639.393988,unique_interior,True,6300500031,中正區,6300500031,1,"[""6300500031""]","[""6300500031""]",1,"[""6300500031""]","[""6300500031""]",True,True,False,True,True,False,consistent,True,conservative_success,0.0,True,unique_interior,True,6300500031,中正區,63000050031,1,"[""6300500031""]","[""63000050031""]",1,"[""6300500031""]","[""63000050031""]",True,True,False,True,True,False,consistent,True,conservative_success,0.0,True,True,6300500031,True,6300500031,True,6300500031,True,6300500031,2024,63000050031,True,False,F

### Summary Stats

品質與完整性摘要；不作統計推論、正式EDA／Finding或里事故組成比例。缺值分母／不可比較狀態分開，不將NA當False。

In [5]:
dictionary=pd.DataFrame(contract['columns'])
dictionary['NA_N']=[int(df[c].isna().sum()) for c in dictionary.name]
dictionary['empty_string_N']=[int(df[c].eq('').sum()) if str(df[c].dtype)=='string' else 0 for c in dictionary.name]
show('完整欄位字典、單位／dtype／來源／NA／空字串',dictionary,block=12)
numeric=[r['name'] for r in contract['columns'] if r['dtype'] in ['Int64','Float64']]
show('數值摘要（有效N／mean／sd／min／q25／q50／q75／max）',df[numeric].describe(percentiles=[.25,.5,.75]).T.reset_index())


**完整欄位字典、單位／dtype／來源／NA／空字串**

,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
0,record_id,string,研究ID＝來源年＋原檔SHA256＋原CSV邏輯列；不是官方事故ID,not applicable,Unit2.1正式CSV：record_id,NA不補0；原字串空白保留,0,0
1,source_year,Int64,來源檔對應西元年,year,Unit2.1正式CSV：source_year,NA不補0；原字串空白保留,0,0
2,source_row_number,Int64,原CSV邏輯資料列，1起算，不含表頭；非物理行號,row,Unit2.1正式CSV：source_row_number,NA不補0；原字串空白保留,0,0
3,coordinate_x_raw,string,原字串：原始點位 X 數值；範圍近似經度但座標基準未確認,not applicable,Unit2.1正式CSV：coordinate_x_raw,NA不補0；原字串空白保留,0,3
4,coordinate_y_raw,string,原字串：原始點位 Y 數值；範圍近似緯度但座標基準未確認,not applicable,Unit2.1正式CSV：coordinate_y_raw,NA不補0；原字串空白保留,0,3
5,longitude_candidate,Float64,X之經度工作解讀；datum未正式認證，採EPSG4326工作慣例,degree candidate,Unit2.1正式CSV：longitude_candidate,NA不補0；原字串空白保留,3,0
6,latitude_candidate,Float64,Y之緯度工作解讀；datum未正式認證，無CRS轉換或geometry,degree candidate,Unit2.1正式CSV：latitude_candidate,NA不補0；原字串空白保留,3,0
7,coordinate_parse_valid,boolean,X及Y皆可解析為有限數；不是里內或CRS認證,logical flag；True/False/NA依欄定義,Unit2.1正式CSV：coordinate_parse_valid,NA不補0；原字串空白保留,0,0
8,coordinate_missing_flag,boolean,任一原座標空白；不補值,logical flag；True/False/NA依欄定義,Unit2.1正式CSV：coordinate_missing_flag,NA不補0；原字串空白保留,0,0
9,original_district_label_raw,string,原事故區序字串原值，數字部分不映射官方CPTID,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
12,accident_address_raw,string,原事故地址，不改寫、不地理編碼、不表示居民地址,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
13,address_district_name,string,只取地址開頭明確區名，可有臺北市／台北市前綴，不從路名猜測,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,152,0
14,address_district_readable,boolean,地址區名可判讀；False不等於地址衝突,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
15,raw_address_district_conflict,boolean,兩來源文字區名之衝突，與地址對空間衝突分開；不可比較NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,152,0
16,point_valid,boolean,來源旗標及全球範圍有效且投影有限；不是位置正確性認證,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
17,point_easting_3826_m,Float64,工作4326→3826的Easting，無補值,m,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
18,point_northing_3826_m,Float64,工作4326→3826的Northing，無補值,m,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
19,fixed_geometry_status,string,幾何互斥狀態；見schema geometry_states，唯一必須within／covered_by同一且各1,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
20,fixed_geometry_unique,boolean,唯一內部配對成立，單純covered_by唯一不成立,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
21,fixed_geometric_village_id,string,唯一內部時的共同10碼研究ID，否則NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,47,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
24,fixed_within_candidate_count,Int64,within候選數；有效Point無命中0，無Point為NA,candidate count,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
25,fixed_within_candidate_ids,string,排序JSON陣列之共同10碼within候選；[]查無命中，NA未查,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
26,fixed_within_original_codes,string,排序JSON原圖資within碼，歷史11碼保留,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
27,fixed_covered_by_candidate_count,Int64,covered_by候選數；有效Point無命中0，無Point為NA,candidate count,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
28,fixed_covered_by_candidate_ids,string,排序JSON陣列之共同10碼covered_by候選；[]查無命中，NA未查,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
29,fixed_covered_by_original_codes,string,排序JSON原圖資covered_by碼，歷史11碼保留,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
30,fixed_district_comparable,boolean,唯一幾何且原區名可判；此集合才作區名衝突率分母,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
31,fixed_district_match,boolean,原區名與空間區名是否一致；不可比較NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,47,0
32,fixed_district_conflict,boolean,原區名與空間區名明確衝突；不可比較NA，不當無衝突,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,47,0
33,fixed_address_comparable,boolean,唯一幾何且地址區名可判；地址衝突率獨立分母,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
36,fixed_address_status,string,unreadable／not_spatially_comparable／consistent／conflict,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
37,fixed_conservative_eligible,boolean,幾何唯一、原區名可判且一致、地址無明確空間衝突,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
38,fixed_assignment_status,string,完整互斥主狀態，按schema priority定義，含0筆狀態,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
39,fixed_distance_to_original_district_m,Float64,Point距此版本原區名所屬里幾何聯集之距離；僅診斷，不重派,m,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
40,fixed_distance_valid,boolean,該版本距離可計算：Point有效且原區名明確,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
41,historical_geometry_status,string,幾何互斥狀態；見schema geometry_states，唯一必須within／covered_by同一且各1,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
42,historical_geometry_unique,boolean,唯一內部配對成立，單純covered_by唯一不成立,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
43,historical_geometric_village_id,string,唯一內部時的共同10碼研究ID，否則NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,50,0
44,historical_spatial_district,string,僅幾何唯一時之空間區名，其他NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,50,0
45,historical_unique_original_boundary_code,string,唯一內部原圖資碼；固定10碼／歷史11碼，不混用,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,50,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
48,historical_within_original_codes,string,排序JSON原圖資within碼，歷史11碼保留,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
49,historical_covered_by_candidate_count,Int64,covered_by候選數；有效Point無命中0，無Point為NA,candidate count,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
50,historical_covered_by_candidate_ids,string,排序JSON陣列之共同10碼covered_by候選；[]查無命中，NA未查,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
51,historical_covered_by_original_codes,string,排序JSON原圖資covered_by碼，歷史11碼保留,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
52,historical_district_comparable,boolean,唯一幾何且原區名可判；此集合才作區名衝突率分母,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
53,historical_district_match,boolean,原區名與空間區名是否一致；不可比較NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,50,0
54,historical_district_conflict,boolean,原區名與空間區名明確衝突；不可比較NA，不當無衝突,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,50,0
55,historical_address_comparable,boolean,唯一幾何且地址區名可判；地址衝突率獨立分母,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
56,historical_address_match,boolean,地址區名與空間區名一致；不可比較NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,202,0
57,historical_address_conflict,boolean,地址與空間區名明確衝突；未知NA不單獨排除保守配對,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,202,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
60,historical_assignment_status,string,完整互斥主狀態，按schema priority定義，含0筆狀態,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
61,historical_distance_to_original_district_m,Float64,Point距此版本原區名所屬里幾何聯集之距離；僅診斷，不重派,m,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,3,0
62,historical_distance_valid,boolean,該版本距離可計算：Point有效且原區名明確,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
63,eligible_s0,boolean,S0納入資格；S0主，S1／S2／S3敏感，不刪不合格原事故,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
64,village_id_s0,string,S0資格成立之共同10碼研究ID，資格False必NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,918,0
65,eligible_s1,boolean,S1納入資格；S0主，S1／S2／S3敏感，不刪不合格原事故,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
66,village_id_s1,string,S1資格成立之共同10碼研究ID，資格False必NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,47,0
67,eligible_s2,boolean,S2納入資格；S0主，S1／S2／S3敏感，不刪不合格原事故,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0
68,village_id_s2,string,S2資格成立之共同10碼研究ID，資格False必NA,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,897,0
69,eligible_s3,boolean,S3納入資格；S0主，S1／S2／S3敏感，不刪不合格原事故,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,0,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
72,historical_boundary_code,string,歷史幾何唯一時原11碼VILLCODE；仍須共同ID連接,not applicable,Unit2.1／固定與年度歷史Spatial predicates,NA不補0；原字串空白保留,50,0
73,fixed_historical_geometry_assignment_same,boolean,兩NA視同未指派；一NA與有效ID不同；兩有效ID字串相等才一致,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
74,geometry_eligibility_changed,boolean,固定與歷史geometry資格xor，不把NA→ID當兩個有效里之換里,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
75,geometry_assigned_to_unassigned,boolean,固定有效→歷史無法納入（geometry）,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
76,geometry_unassigned_to_assigned,boolean,固定無法納入→歷史有效（geometry）,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
77,fixed_historical_conservative_assignment_same,boolean,兩NA視同未指派；一NA與有效ID不同；兩有效ID字串相等才一致,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
78,conservative_eligibility_changed,boolean,固定與歷史conservative資格xor，不把NA→ID當兩個有效里之換里,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
79,conservative_assigned_to_unassigned,boolean,固定有效→歷史無法納入（conservative）,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
80,conservative_unassigned_to_assigned,boolean,固定無法納入→歷史有效（conservative）,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
81,geometric_village_changed,boolean,兩邊都幾何唯一且研究里碼不同；缺一邊不稱兩有效里換里,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0


,name,dtype,definition,unit,source,missing_rule,NA_N,empty_string_N
84,geometry_village_or_status_changed,boolean,與Unit1.7可比：幾何里碼（未指派作同一空狀態）或幾何狀態變更,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
85,district_or_assignment_availability_changed,boolean,空間區名或其可指派性改變；不冒充皆為有效區換區,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
86,district_conflict_to_consistent,boolean,固定明確區名衝突→歷史明確一致；原239診斷口徑,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
87,district_consistent_to_conflict,boolean,固定明確一致→歷史明確衝突；不含原未配對,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
88,new_district_conflict_including_unassigned,boolean,歷史新衝突，包括固定未可比較者；與原215診斷口徑對照,not applicable,四種正式規格逐筆比較；NA語義明示,NA不補0；原字串空白保留,0,0
89,geometry_change_reason,string,換里＝兩邊幾何唯一ID不同；資格變更＝geometry資格xor；四類互斥,not applicable,固定／年度歷史比較,NA不補0；原字串空白保留,0,0
90,conservative_change_reason,string,換里＝兩邊幾何唯一ID不同；資格變更＝conservative資格xor；四類互斥,not applicable,固定／年度歷史比較,NA不補0；原字串空白保留,0,0


**數值摘要（有效N／mean／sd／min／q25／q50／q75／max）**

,index,count,mean,std,min,25%,50%,75%,max
0,source_year,45130.0,2024.504365,0.499986,2024.0,2024.0,2025.0,2025.0,2025.0
1,source_row_number,45130.0,18589.632971,13615.934151,1.0,7879.0,15770.0,25731.5,51809.0
2,longitude_candidate,45127.0,121.540589,0.030696,121.46118,121.516905,121.536807,121.562727,121.652018
3,latitude_candidate,45127.0,25.053776,0.032341,24.96593,25.034177,25.049516,25.068785,25.188246
4,point_easting_3826_m,45127.0,304543.955358,3099.967582,296499.240275,302154.450914,304164.784393,306785.154958,315800.812094
5,point_northing_3826_m,45127.0,2771843.196516,3579.850751,2762130.785375,2769671.221318,2771378.162303,2773519.191854,2786750.248707
6,fixed_within_candidate_count,45127.0,0.999025,0.031211,0.0,1.0,1.0,1.0,1.0
7,fixed_covered_by_candidate_count,45127.0,0.999025,0.031211,0.0,1.0,1.0,1.0,1.0
8,fixed_distance_to_original_district_m,45127.0,6.214102,127.616513,0.0,0.0,0.0,0.0,6409.806107
9,historical_within_candidate_count,45127.0,0.998958,0.032256,0.0,1.0,1.0,1.0,1.0


四規格及互斥狀態優先序：S0（主要）：固定界，within與covered_by各唯一且同里，來源區名可確認並一致，地址沒有明確空間衝突。S1：固定界幾何唯一，保留區名／地址衝突。S2：對應年度歷史界＋同S0保守條件。S3：對應年度歷史界＋同S1幾何條件。S0⊆S1、S2⊆S3；同家族共同有效ID必相同，敏感規格不自動升主分析。 先處理缺／無效座標與幾何例外，再判原區名未知／衝突、額外地址衝突，最後保守成功；獨立旗標另保留。

In [6]:
show('S0–S3完整定義',pd.DataFrame(list(contract['assignment_specifications'].items()),columns=['specification','definition']))
show('十個完整互斥狀態優先序（含0筆）',pd.DataFrame(list(contract['assignment_priority'].items()),columns=['status','priority_definition']))
stats=[];states=[]
for year,g in assignment.groupby('source_year'):
 for spec in ['s0','s1','s2','s3']:stats.append(dict(year=int(year),specification=spec,candidates=len(g),eligible_N=int(g['eligible_'+spec].sum()),eligible_rate=float(g['eligible_'+spec].sum()/len(g))))
 for family in ['fixed','historical']:
  for state in STATES:states.append(dict(year=int(year),boundary=family,status=state,N=int(g[family+'_assignment_status'].eq(state).sum())))
show('各年四規格N／比例，分母為各年全候選',pd.DataFrame(stats))
show('固定及歷史互斥狀態全表',pd.DataFrame(states),block=10)
show('固定／歷史變化分類',assignment.groupby(['source_year','conservative_change_reason']).size().rename('N').reset_index())
show('主規格未納入前20筆；全部仍保存在正式CSV',assignment.loc[~assignment.eligible_s0,['record_id','source_year','original_district_name','address_district_name','fixed_spatial_district','fixed_assignment_status','fixed_distance_to_original_district_m','village_id_s1','village_id_s2','village_id_s3']].head(20),block=10)
show('三筆缺座標，四規格均未強派',assignment.loc[assignment.coordinate_missing_flag,['record_id','source_year','coordinate_x_raw','coordinate_y_raw','fixed_assignment_status','historical_assignment_status','eligible_s0','eligible_s1','eligible_s2','eligible_s3']])


**S0–S3完整定義**

,specification,definition
0,s0,PRIMARY: fixed geometry unique within=covered_by same sole village; original district readable and consist...
1,s1,SENSITIVITY: fixed geometry unique only; district/address conflicts retained
2,s2,SENSITIVITY: source-year historical boundary plus same conservative tests as S0
3,s3,SENSITIVITY: source-year historical geometry unique only


**十個完整互斥狀態優先序（含0筆）**

,status,priority_definition
0,missing_coordinates,1 原座標缺失，不建立Point
1,invalid_coordinates,2 非缺失但不可解析／非有限／候選經緯度越界或投影失敗
2,outside_or_hole,3 有效Point的within及covered_by均無候選
3,boundary_nonunique,4 共同邊界／頂點或單一邊界；不是唯一內部
4,multiple_geometry,5 多個內部或其他多重幾何候選，不任選第一筆
5,predicate_mismatch,6 within非covered_by子集或其他不一致
6,original_district_unknown,7 幾何唯一但來源區名無法確認
7,district_conflict,8 幾何唯一但來源區名與空間區名衝突
8,address_conflict,9 幾何唯一且來源區名一致，但可判讀地址與空間區名衝突
9,conservative_success,10 唯一內部、原區名明確一致、地址沒有明確空間衝突


**各年四規格N／比例，分母為各年全候選**

,year,specification,candidates,eligible_N,eligible_rate
0,2024,s0,22368,21920,0.979971
1,2024,s1,22368,22351,0.999240
2,2024,s2,22368,21931,0.980463
3,2024,s3,22368,22350,0.999195
4,2025,s0,22762,22292,0.979352
5,2025,s1,22762,22732,0.998682
6,2025,s2,22762,22302,0.979791
7,2025,s3,22762,22730,0.998594


... (8 outputs left out)

,record_id,source_year,original_district_name,address_district_name,fixed_spatial_district,fixed_assignment_status,fixed_distance_to_original_district_m,village_id_s1,village_id_s2,village_id_s3
232,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000554,2024,大安區,大安區,信義區,district_conflict,0.04166,6300200040,6300300048,6300300048
296,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000701,2024,萬華區,萬華區,中正區,district_conflict,1363.886157,6300500028,<NA>,6300500028
345,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000814,2024,大安區,大安區,松山區,district_conflict,1.768911,6300100031,6300300054,6300300054
403,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000955,2024,中正區,中正區,大安區,district_conflict,7.747477,6300300057,<NA>,6300300057
458,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r001082,2024,內湖區,內湖區,<NA>,outside_or_hole,998.444989,<NA>,<NA>,<NA>
551,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r001298,2024,松山區,松山區,信義區,district_conflict,9.983809,6300200005,<NA>,6300200005
590,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r001388,2024,大安區,大安區,松山區,district_conflict,1.410798,6300100032,6300300054,6300300054
690,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r001623,2024,萬華區,萬華區,大同區,district_conflict,370.986322,6300600001,<NA>,6300600001
727,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r001709,2024,松山區,松山區,信義區,district_conflict,56.682313,6300200014,<NA>,6300200014
770,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r001803,2024,松山區,松山區,大安區,district_conflict,2.226426,6300300049,<NA>,6300300049


,record_id,source_year,original_district_name,address_district_name,fixed_spatial_district,fixed_assignment_status,fixed_distance_to_original_district_m,village_id_s1,village_id_s2,village_id_s3
794,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r001855,2024,大同區,大同區,中正區,district_conflict,8.262834,6300500024,6300600002,6300600002
816,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r001905,2024,大安區,大安區,中正區,district_conflict,27.183768,6300500019,<NA>,6300500019
927,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r002167,2024,大同區,大同區,<NA>,outside_or_hole,19.041441,<NA>,<NA>,<NA>
981,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r002301,2024,信義區,信義區,大安區,district_conflict,3.847392,6300300034,<NA>,6300300034
1258,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r002923,2024,文山區,文山區,中正區,district_conflict,2.226814,6300500001,6300800010,6300800010
1308,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r003032,2024,萬華區,萬華區,中正區,district_conflict,0.010191,6300500014,6300700014,6300700014
1400,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r003240,2024,萬華區,萬華區,中正區,district_conflict,22.484424,6300500022,<NA>,6300500022
1512,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r003503,2024,萬華區,萬華區,中正區,district_conflict,97.821609,6300500016,<NA>,6300500016
1553,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r003591,2024,萬華區,萬華區,中正區,district_conflict,32.996349,6300500023,<NA>,6300500023
1598,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r003687,2024,中山區,中山區,中正區,district_conflict,5.908155,6300500025,<NA>,6300500025


**三筆缺座標，四規格均未強派**

,record_id,source_year,coordinate_x_raw,coordinate_y_raw,fixed_assignment_status,historical_assignment_status,eligible_s0,eligible_s1,eligible_s2,eligible_s3
16594,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r038470,2024,,,missing_coordinates,missing_coordinates,False,False,False,False
26801,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r004434,2025,,,missing_coordinates,missing_coordinates,False,False,False,False
43928,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r021561,2025,,,missing_coordinates,missing_coordinates,False,False,False,False


### Assertions

獨立保存檔驗證：456里／45,130事故、dtype、完整鍵集合、四規格foreign keys／子集／共同有效ID、候選JSON與NA語義、原座標／來源未改，以及與正式事故一對一銜接。

In [7]:
assert validate_saved_pair(base,assignment,upstream)
for table,con in [(base,base_contract),(assignment,assignment_contract)]:
 assert len(table)==con['expected_rows'] and list(table)==[r['name'] for r in con['columns']]
 for field in con['columns']:assert str(table[field['name']].dtype)==field['dtype']
assert assignment.groupby('source_year')[['eligible_s0','eligible_s1','eligible_s2','eligible_s3']].sum().to_dict()=={'eligible_s0':{2024:21920,2025:22292},'eligible_s1':{2024:22351,2025:22732},'eligible_s2':{2024:21931,2025:22302},'eligible_s3':{2024:22350,2025:22730}}
assert assignment.fixed_assignment_status.eq('district_conflict').sum()==869
assert assignment.fixed_assignment_status.eq('address_conflict').sum()==2
assert assignment.fixed_assignment_status.eq('outside_or_hole').sum()==44
assert assignment.geometry_village_or_status_changed.groupby(assignment.source_year).sum().to_dict()=={2024:544,2025:585}
assert assignment.district_conflict_to_consistent.sum()==239 and assignment.new_district_conflict_including_unassigned.sum()==215
assert set(assignment.fixed_assignment_status).issubset(STATES)
assert set(assignment.historical_assignment_status).issubset(STATES)
# Cardinality-only technical handoff, no feature aggregation or serialized result.
joined=upstream[['record_id','source_year']].merge(assignment[['record_id','eligible_s0','village_id_s0','eligible_s1','village_id_s1']],on='record_id',how='left',validate='one_to_one')
assert len(joined)==45130 and joined.eligible_s0.notna().all()
village_join=joined.merge(base[['village_id','standard_village_name','village_area_km2']],left_on='village_id_s0',right_on='village_id',how='left',validate='many_to_one')
assert len(village_join)==45130 and village_join.standard_village_name.notna().sum()==44212
skeleton=pd.MultiIndex.from_product([base.village_id.tolist(),[2024,2025]],names=['village_id','year']).to_frame(index=False)
assert len(skeleton)==912 and not skeleton.duplicated(['village_id','year']).any() and base.village_id.nunique()==456
show('Unit2.3銜接使用範例：原事故與配對一對一，再接主里，不輸出新資料',village_join.head(10),block=10)
print('可形成912 village-year及456兩年共同里骨架；本文件未計正式Dataset A／C比例。')
for relative,expected in EXPECTED_OUTPUT_HASHES.items():assert sha(PROJECT_ROOT/relative)==expected
for relative,expected in contract['source_inputs'].items():assert sha(PROJECT_ROOT/relative)==expected


**Unit2.3銜接使用範例：原事故與配對一對一，再接主里，不輸出新資料**

,record_id,source_year,eligible_s0,village_id_s0,eligible_s1,village_id_s1,village_id,standard_village_name,village_area_km2
0,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000001,2024,True,6300200006,True,6300200006,6300200006,興雅里,0.248822
1,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000003,2024,True,6300500031,True,6300500031,6300500031,三愛里,0.215015
2,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000006,2024,True,6300100024,True,6300100024,6300100024,新聚里,0.14936
3,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000007,2024,True,6300100025,True,6300100025,6300100025,復盛里,0.148935
4,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000009,2024,True,6300300039,True,6300300039,6300300039,臥龍里,0.195223
5,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000011,2024,True,6300100002,True,6300100002,6300100002,莊敬里,1.281723
6,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000014,2024,True,6300400003,True,6300400003,6300400003,正得里,0.146164
7,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000016,2024,True,6300500001,True,6300500001,6300500001,水源里,0.410551
8,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000019,2024,True,6300600009,True,6300600009,6300600009,大有里,0.207423
9,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000025,2024,True,6301100025,True,6301100025,6301100025,岩山里,0.491043


可形成912 village-year及456兩年共同里骨架；本文件未計正式Dataset A／C比例。


In [8]:
result={'status':'passed','kind':'assignment','rows':len(df),'columns':len(df.columns),'sha256':'2abcb5a69c8998c2f8b40c975404b93d629fd725899f74afc991f349d9201ba5','schema_sha256':'d43e0e3bcc1af2b7683d9466ccb4c441c39a6d9ce56f957b0ed5a89c1b487ad3','independent_saved_file_read':True,'all_45130_candidates_retained':True,'all_456_villages_retained':True,'human_acceptance':'Codex協助；DA／PM待審'}
(PROJECT_ROOT/'data/temp/unit22_validation/assignment_documentation_summary.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([result]))

,status,kind,rows,columns,sha256,schema_sha256,independent_saved_file_read,all_45130_candidates_retained,all_456_villages_retained,human_acceptance
0,passed,assignment,45130,91,2abcb5a69c8998c2f8b40c975404b93d629fd725899f74afc991f349d9201ba5,d43e0e3bcc1af2b7683d9466ccb4c441c39a6d9ce56f957b0ed5a89c1b487ad3,True,True,True,Codex協助；DA／PM待審


技術交付已核對，實際成員貢獻／DA驗收／PM審查另待完成。後續以本文件固定CSV＋schema＋hash讀取；主規格S0，敏感規格不自動升格，也不要求每個模型重跑四遍。事故特徵合併比例由分子／有效分母重算，不平均年度比例；設籍人口與道路供給均非交通暴露。完成本Unit後停止，未執行Unit2.3或其他正式里特徵。